# ComfyUI Krea2 — Kaggle T4 / TPU 共用 Notebook

一份 Notebook，使用下方選單選擇 T4 或 TPU，接著依序執行。**選單不會改變 Kaggle 分配的硬體**：先在右側 Settings → Accelerator 選 GPU T4 x2 或 TPU；變更加速器會重新啟動工作階段，請重新執行選單與後續儲存格。若卡在 Draft Session Starting，請先等待 Kaggle 配置完成，選單要等工作階段啟動後才可使用。

本 Notebook 只下載既有 Krea2 Neutrino 權重、Qwen3VL 編碼器及 Qwen VAE。TPU 與 T4 上完整生成均尚未驗證；NVFP4 在 T4 上的相容性也未驗證。下方圖片生成儲存格會顯示實際結果或錯誤，不代表保證成功。模型下載需要啟用 Kaggle Internet 並有足夠磁碟空間。

操作：1. 設定加速器及 Internet → 2. 執行設定儲存格、選擇對應模式 → 3. 依序執行後續儲存格。要再次生成可修改提示詞並重新執行生成儲存格；最後的停止儲存格只在結束使用時執行。


In [ ]:
import hashlib
import json
import os
import pathlib
import platform
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import uuid
import venv

BASE = pathlib.Path('/kaggle/working')
BASE.mkdir(parents=True, exist_ok=True)
APP = BASE / 'ComfyUI_Krea2'
MODE = 'gpu'  # Default for Run All; the selector below can change this for interactive runs.
TPU_ENV_MODE = 'venv'  # 'venv' installs pinned XLA packages in isolation; 'existing' requires the exact recorded tuple in the kernel.
assert MODE in ('tpu_experimental', 'gpu')
assert MODE != 'tpu_experimental' or TPU_ENV_MODE in ('venv', 'existing')

env_tag = TPU_ENV_MODE if MODE == 'tpu_experimental' else 'gpu-inherited'
RUNTIME_ENV_DIR = BASE / f'comfyui-runtime-{env_tag}-py{sys.version_info.major}{sys.version_info.minor}'
RUNTIME_PYTHON = RUNTIME_ENV_DIR / 'bin' / 'python'
RUNTIME_MARKER = RUNTIME_ENV_DIR / '.codex-runtime.json'
TPU_EXPECTED = {'torch': '2.9.0+cpu', 'torchvision': '0.24.0+cpu', 'torch_xla': '2.9.0', 'libtpu': '0.0.21.1'}

PROMPT = 'A quiet seaside village at blue hour, warm window lights reflected on calm water, cinematic composition, intricate natural details.'
SEED = 271828
WIDTH = 1024
HEIGHT = 1024
STEPS = 10
CFG = 1.0

nvidia_smi = shutil.which('nvidia-smi')
gpu_report = subprocess.run([nvidia_smi, '-L'], capture_output=True, text=True).stdout.strip() if nvidia_smi else 'Not detected (nvidia-smi unavailable)'
print('Jupyter kernel Python:', sys.version.split()[0])
print('Working directory:', BASE)
print('MODE:', MODE)
print('TPU_ENV_MODE:', TPU_ENV_MODE if MODE == 'tpu_experimental' else 'not used')
print('Runtime venv:', RUNTIME_ENV_DIR)
print('GPU:', gpu_report)


In [ ]:
# 互動選單：選好後執行下一格。若使用 Run All，預設採 T4 模式。
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as exc:
    raise RuntimeError('此介面需要 ipywidgets，請確認 Kaggle Internet 已開啟後安裝 ipywidgets。') from exc

accelerator_selector = widgets.ToggleButtons(
    options=[('NVIDIA T4', 'gpu'), ('TPU', 'tpu_experimental')],
    value='gpu', description='執行模式:',
)
prompt_input = widgets.Textarea(value=PROMPT, description='提示詞:', layout=widgets.Layout(width='95%', height='85px'))
display(accelerator_selector, prompt_input)
print('選單只選擇程式流程；實際硬體請在 Kaggle Settings → Accelerator 設定，切換硬體後重新執行本格。')


In [ ]:
# 套用選單並準備獨立路徑；若變更選擇，請從此格起重新執行。
MODE = accelerator_selector.value
PROMPT = prompt_input.value.strip()
if not PROMPT:
    raise ValueError('請輸入提示詞。')
APP = BASE / ('ComfyUI_Krea2_T4' if MODE == 'gpu' else 'ComfyUI_Krea2_TPU')
env_tag = 'gpu-inherited' if MODE == 'gpu' else TPU_ENV_MODE
RUNTIME_ENV_DIR = BASE / f'comfyui-runtime-{env_tag}-py{sys.version_info.major}{sys.version_info.minor}'
RUNTIME_PYTHON = RUNTIME_ENV_DIR / 'bin' / 'python'
RUNTIME_MARKER = RUNTIME_ENV_DIR / '.codex-runtime.json'
print('已選擇:', 'T4' if MODE == 'gpu' else 'TPU', '| ComfyUI:', APP)
print('確認 Kaggle Settings → Accelerator 已選相同硬體，否則接下來的檢查會停止。')


## 1. Select and prepare an isolated runtime

For TPU mode, TPU_ENV_MODE='venv' creates a dedicated environment under /kaggle/working and installs the exact torch 2.9.0+cpu, torch_xla 2.9.0, and libtpu 0.0.21.1 tuple there. All XLA checks and the ComfyUI server run as child processes from that environment. The Jupyter kernel's installed packages and environment variables are left alone.

Choose TPU_ENV_MODE='existing' only if the kernel already reports the exact previously recorded Colab tuple: Python 3.13.18, torch 2.9.0+cpu, torch_xla 2.9.0, and libtpu 0.0.21.1. This mode still launches ComfyUI from a venv and does not pip-install into the kernel.

The pinned PyTorch/XLA release supports Python 3.10–3.13, while the exact libtpu 0.0.21.1 wheel is available for CPython 3.11–3.14; this notebook permits only their Linux x86_64 intersection (3.11–3.13). Python 3.10 and 3.14 or newer stop with a specific explanation; the notebook never installs or downgrades Python. Only Python 3.13.18 has prior end-to-end XLA runtime evidence, and no Python version has proven Krea 2 image generation here.

GPU mode also uses a venv with Kaggle's existing GPU packages visible as system packages. The runtime check below reports versions and fails closed on tuple mismatches.


In [ ]:
if MODE == 'gpu':
    gpu_check = subprocess.run([nvidia_smi, '-L'], capture_output=True, text=True) if nvidia_smi else None
    if gpu_check is None or gpu_check.returncode != 0:
        raise RuntimeError('請在 Kaggle Settings → Accelerator 選擇 GPU T4 x2，等待工作階段啟動後重新執行。')
    if 'T4' not in gpu_check.stdout.upper():
        raise RuntimeError('目前 GPU 並非 T4：' + gpu_check.stdout.strip() + '。請在 Kaggle 選擇 GPU T4 x2。')
    print('已偵測 T4:', gpu_check.stdout.strip())
else:
    if sys.platform != 'linux' or platform.machine().lower() not in ('x86_64', 'amd64'):
        raise RuntimeError(f'The pinned TPU wheels require Linux x86_64; detected {sys.platform}/{platform.machine()}.')
    supported_python = {(3, 11), (3, 12), (3, 13)}
    if sys.version_info[:2] not in supported_python:
        raise RuntimeError(
            f'Kaggle is running Python {sys.version.split()[0]}. This pinned workflow needs CPython 3.11–3.13: '
            'PyTorch/XLA 2.9 documents 3.10–3.13, but libtpu 0.0.21.1 has no CPython 3.10 wheel. '
            'The notebook will not install or downgrade Python; choose a Kaggle runtime using CPython 3.11, 3.12, or 3.13.'
        )
    glibc_text = platform.libc_ver()[1]
    try:
        glibc_parts = tuple(int(part) for part in glibc_text.split('.')[:2])
    except ValueError:
        glibc_parts = ()
    if not glibc_parts or glibc_parts < (2, 31):
        raise RuntimeError(f'libtpu 0.0.21.1 wheels require glibc 2.31 or newer; detected {glibc_text or "unknown"}.')

    if TPU_ENV_MODE == 'existing':
        import importlib.metadata
        kernel_tuple = {}
        for distribution in ('torch', 'torchvision', 'torch_xla', 'libtpu'):
            try:
                kernel_tuple[distribution] = importlib.metadata.version(distribution)
            except importlib.metadata.PackageNotFoundError:
                kernel_tuple[distribution] = 'missing'
        print('Jupyter kernel TPU tuple:', {'python': sys.version.split()[0], **kernel_tuple})
        if sys.version_info[:3] != (3, 13, 18) or kernel_tuple != TPU_EXPECTED:
            raise RuntimeError(
                'TPU_ENV_MODE=existing requires the exact recorded tuple (Python 3.13.18, torch 2.9.0+cpu, '
                'torchvision 0.24.0+cpu, torch_xla 2.9.0, libtpu 0.0.21.1). Switch TPU_ENV_MODE to venv to install the pinned tuple in isolation.'
            )
    elif sys.version_info[:3] != (3, 13, 18):
        print('UNVERIFIED PYTHON MINOR/MICRO: package wheels are within the documented range, but the prior XLA runtime probe used Python 3.13.18.')

system_site_packages = (MODE == 'gpu' or (MODE == 'tpu_experimental' and TPU_ENV_MODE == 'existing'))
runtime_config = {
    'mode': MODE,
    'tpu_env_mode': TPU_ENV_MODE if MODE == 'tpu_experimental' else None,
    'base_python': list(sys.version_info[:3]),
    'base_executable': sys.executable,
    'system_site_packages': system_site_packages,
}
if RUNTIME_ENV_DIR.exists():
    if not RUNTIME_MARKER.is_file():
        raise RuntimeError(f'Runtime path exists without this notebook ownership marker: {RUNTIME_ENV_DIR}. Set a fresh RUNTIME_ENV_DIR before continuing; it will not be deleted.')
    previous_config = json.loads(RUNTIME_MARKER.read_text())
    if previous_config != runtime_config or not RUNTIME_PYTHON.is_file():
        raise RuntimeError(f'Existing runtime venv does not match the selected Kaggle runtime: {RUNTIME_ENV_DIR}. Use a fresh RUNTIME_ENV_DIR; it will not be modified.')
else:
    RUNTIME_ENV_DIR.parent.mkdir(parents=True, exist_ok=True)
    venv.EnvBuilder(with_pip=True, system_site_packages=system_site_packages).create(RUNTIME_ENV_DIR)
    RUNTIME_MARKER.write_text(json.dumps(runtime_config, indent=2) + '\n')

runtime_probe = r'''import importlib.metadata, json, platform, sys
versions = {}
for name in ('torch', 'torchvision', 'torch_xla', 'libtpu'):
    try:
        versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        versions[name] = 'missing'
print(json.dumps({'python': platform.python_version(), 'versions': versions}))
'''
probe = subprocess.run([str(RUNTIME_PYTHON), '-c', runtime_probe], capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError('Could not inspect the isolated runtime Python:\n' + probe.stderr[-4000:])
runtime_info = json.loads(probe.stdout.strip())
if runtime_info['python'].split('.')[:2] != [str(sys.version_info.major), str(sys.version_info.minor)]:
    raise RuntimeError(f'The venv Python does not match Kaggle kernel Python: {runtime_info}')

if MODE == 'tpu_experimental' and TPU_ENV_MODE == 'venv' and runtime_info['versions'] != TPU_EXPECTED:
    pip = [str(RUNTIME_PYTHON), '-m', 'pip', 'install', '--upgrade']
    subprocess.run(
        pip + ['--index-url', 'https://pypi.org/simple', '--extra-index-url', 'https://download.pytorch.org/whl/cpu', 'torch==2.9.0+cpu', 'torchvision==0.24.0+cpu'],
        check=True,
    )
    subprocess.run(
        pip + ['--find-links', 'https://storage.googleapis.com/libtpu-wheels/index.html', 'torch_xla==2.9.0', 'libtpu==0.0.21.1'],
        check=True,
    )
    probe = subprocess.run([str(RUNTIME_PYTHON), '-c', runtime_probe], capture_output=True, text=True, check=True)
    runtime_info = json.loads(probe.stdout.strip())

print('Runtime Python:', str(RUNTIME_PYTHON))
print('Runtime tuple:', runtime_info)
if MODE == 'tpu_experimental' and runtime_info['versions'] != TPU_EXPECTED:
    raise RuntimeError(f'Isolated TPU runtime did not resolve the exact pinned package tuple: {runtime_info}')
if MODE == 'tpu_experimental':
    print('PJRT_DEVICE=TPU will be set only in the XLA smoke-test and ComfyUI child-process environments.')
else:
    print('GPU runtime inherits Kaggle packages through the venv; the Jupyter kernel is not modified.')

if MODE == 'gpu':
    gpu_probe = subprocess.run([str(RUNTIME_PYTHON), '-c', 'import torch; print(torch.__version__); print(torch.cuda.get_device_name(0)); assert torch.cuda.is_available()'], capture_output=True, text=True)
    if gpu_probe.returncode != 0:
        raise RuntimeError('T4 的 PyTorch CUDA 檢查失敗：\n' + gpu_probe.stdout + gpu_probe.stderr[-4000:])
    print('T4 PyTorch CUDA 檢查通過：', gpu_probe.stdout.strip())


## 2. Get modern ComfyUI

This uses the official ComfyUI repository, rather than the older ComfyUI-XLA fork, so the Krea 2 loader path is available. The XLA patch below must match the pinned ComfyUI revision.


In [ ]:
COMFYUI_URL = 'https://github.com/Comfy-Org/ComfyUI.git'
# Set to the exact upstream revision supported by the shared XLA patch before publishing.
COMFYUI_COMMIT = 'e9afbccc2669677076feb08e52658f129cd8c12b'

if not APP.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--depth', '1', COMFYUI_URL, str(APP)], check=True)
if not (APP / '.git').exists():
    raise RuntimeError(f'{APP} exists but is not an official ComfyUI Git checkout; choose an empty working directory.')
current_commit = subprocess.run(['git', '-C', str(APP), 'rev-parse', 'HEAD'], capture_output=True, text=True)
if current_commit.returncode != 0:
    subprocess.run(['git', '-C', str(APP), 'fetch', '--depth', '1', 'origin', COMFYUI_COMMIT], check=True)
    subprocess.run(['git', '-C', str(APP), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
else:
    current_sha = current_commit.stdout.strip()
    if current_sha != COMFYUI_COMMIT:
        dirty = subprocess.run(['git', '-C', str(APP), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout.strip()
        if dirty:
            raise RuntimeError(f'Existing ComfyUI checkout has local changes at {current_sha}; refusing to switch revisions.')
        subprocess.run(['git', '-C', str(APP), 'fetch', '--depth', '1', 'origin', COMFYUI_COMMIT], check=True)
        subprocess.run(['git', '-C', str(APP), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
print('ComfyUI commit:', subprocess.run(['git', '-C', str(APP), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip())


## 3. Install ComfyUI dependencies and apply the shared XLA patch

The common patch matches the pinned official ComfyUI revision and adds the --xla launch flag plus XLA-aware device and memory handling; it does not implement multi-core sharding. The exact patch bytes are embedded below and checked by SHA-256 before application.

For TPU mode, the pinned PyTorch/XLA tuple is installed only in the dedicated venv. ComfyUI requirements are installed into that same venv after filtering out PyTorch, XLA, libtpu, and xformers packages, so the accelerator tuple is protected. No pip command runs against the Jupyter kernel. In existing mode the pre-existing exact tuple is visible through a system-site-packages venv but remains unmodified.

For GPU mode, keep Kaggle's supplied PyTorch/CUDA packages visible through the venv and do not apply the TPU patch.


In [ ]:
# The exact shared XLA patch snapshot is embedded here and SHA-256 checked against its source artifact.
# The runtime-version guard ran in cell 3. No torch/torch_xla/libtpu pip install occurs.
if MODE == 'tpu_experimental':
    import base64
    PATCH_B64 = (
    '...'
)
    PATCH_BYTES = base64.b64decode(''.join(PATCH_B64.split()))
    PATCH_SHA256 = '52c0d818db799db68f3053f0f67d726171c22c1a9e6bf2daa48d38e8dc1ddf7b'
    patch_digest = hashlib.sha256(PATCH_BYTES).hexdigest()
    if patch_digest != PATCH_SHA256:
        raise RuntimeError(f'Embedded XLA patch hash mismatch: {patch_digest}')
    if subprocess.run(['git', '-C', str(APP), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip() != COMFYUI_COMMIT:
        raise RuntimeError('The shared XLA patch requires the pinned ComfyUI commit.')
    patch_path = BASE / 'comfyui_xla_29.patch'
    patch_path.write_bytes(PATCH_BYTES)
    patch_marker = APP / '.codex_comfyui_xla_29.sha256'
    if patch_marker.is_file() and patch_marker.read_text().strip() == PATCH_SHA256:
        print('Verified XLA patch is already applied.')
    else:
        check = subprocess.run(['git', '-C', str(APP), 'apply', '--check', str(patch_path)], capture_output=True, text=True)
        if check.returncode == 0:
            subprocess.run(['git', '-C', str(APP), 'apply', str(patch_path)], check=True)
        else:
            reverse = subprocess.run(['git', '-C', str(APP), 'apply', '--reverse', '--check', str(patch_path)], capture_output=True, text=True)
            if reverse.returncode != 0:
                raise RuntimeError('Shared XLA patch does not apply cleanly to the pinned ComfyUI checkout.\n' + check.stderr)
            print('Patch changes are already present in the pinned checkout.')
        patch_marker.write_text(PATCH_SHA256 + '\n')
    print('Applied shared XLA patch:', PATCH_SHA256)

req_file = APP / 'requirements.txt'
if not req_file.is_file():
    raise FileNotFoundError(f'Missing ComfyUI requirements: {req_file}')
import re
reqs = []
protected_packages = {'torch', 'torchvision', 'torchaudio', 'torch-xla', 'libtpu', 'xformers'}
for raw in req_file.read_text().splitlines():
    line = raw.strip()
    if not line or line.startswith('#'):
        continue
    match = re.match(r'([A-Za-z0-9_.-]+)', line)
    package_name = match.group(1).lower().replace('_', '-') if match else ''
    if package_name in protected_packages:
        continue
    reqs.append(line)
filtered = BASE / 'requirements-no-accelerator.txt'
filtered.write_text('\n'.join(reqs) + '\n')
install_command = [str(RUNTIME_PYTHON), '-m', 'pip', 'install', '-r', str(filtered)]
if MODE == 'tpu_experimental':
    constraints = BASE / 'tpu-package-constraints.txt'
    constraints.write_text('torch==2.9.0+cpu\ntorchvision==0.24.0\ntorch_xla==2.9.0\nlibtpu==0.0.21.1\n')
    install_command.extend(['--constraint', str(constraints)])
subprocess.run(install_command, check=True)
probe = subprocess.run([str(RUNTIME_PYTHON), '-c', runtime_probe], capture_output=True, text=True, check=True)
runtime_info = json.loads(probe.stdout.strip())
if MODE == 'tpu_experimental' and runtime_info['versions'] != TPU_EXPECTED:
    raise RuntimeError(f'ComfyUI dependencies changed the pinned TPU tuple inside its venv: {runtime_info}')
print('Installed ComfyUI dependencies inside:', RUNTIME_ENV_DIR)
print('Final runtime tuple:', runtime_info)


## 4. Run a TPU smoke test in the isolated runtime

This executes a small matrix operation in a child process launched with the venv Python and PJRT_DEVICE=TPU. A passing result proves only that this Python/XLA process can see a TPU and execute this operation; it does not prove that ComfyUI, the NVFP4 checkpoint, or Krea 2 image generation works.


In [ ]:
if MODE == 'tpu_experimental':
    smoke_code = r'''import json, torch
import torch_xla.core.xla_model as xm
device = xm.xla_device()
result = torch.ones((32, 32), device=device) @ torch.ones((32, 32), device=device)
xm.mark_step()
print('TPU_SMOKE_RESULT=' + json.dumps({'device': str(device), 'value': result.cpu()[0, 0].item()}))
'''
    child_env = os.environ.copy()
    child_env['PJRT_DEVICE'] = 'TPU'
    smoke = subprocess.run([str(RUNTIME_PYTHON), '-c', smoke_code], env=child_env, capture_output=True, text=True)
    print(smoke.stdout)
    if smoke.stderr:
        print(smoke.stderr[-8000:])
    if smoke.returncode != 0 or 'TPU_SMOKE_RESULT=' not in smoke.stdout:
        raise RuntimeError(f'Isolated TPU smoke test failed with status {smoke.returncode}. Kaggle TPU execution is not verified.')
else:
    gpu_code = "import json, torch; assert torch.cuda.is_available(); print('GPU_SMOKE_RESULT=' + json.dumps({'device': torch.cuda.get_device_name(0)}))"
    gpu = subprocess.run([str(RUNTIME_PYTHON), '-c', gpu_code], capture_output=True, text=True)
    print(gpu.stdout)
    if gpu.stderr:
        print(gpu.stderr[-8000:])
    if gpu.returncode != 0:
        raise RuntimeError('CUDA is unavailable in the isolated GPU runtime.')


## 5. Download only the existing Krea 2 model profile

The three pinned files are the existing Neutrino Krea 2 checkpoint, its Qwen3VL encoder, and the Qwen image VAE. Each file is checked against the manifest's exact size and SHA-256 before ComfyUI uses it. A mismatch stops the notebook.


In [ ]:
MODEL_MANIFEST = [
    {
        'path': 'diffusion_models/Neutrino_v2_base_nvfp4_svd.safetensors',
        'size': 8167362324,
        'url': 'https://huggingface.co/Crowlley/Krea2Neutrino/resolve/347783b20f638e3ee5984607db5f1a51939f8afa/Neutrino_v2_base_nvfp4_svd.safetensors',
        'sha256': '6844374f31e7de278c3408b6333b8fbb7bf5cdd1321646d5480bf1aeda683e1a',
    },
    {
        'path': 'text_encoders/qwen3vl_4b_fp8_scaled.safetensors',
        'size': 5242467968,
        'url': 'https://huggingface.co/Comfy-Org/Krea-2/resolve/1171f4dc7e325bf3734586942e737872506c73e1/text_encoders/qwen3vl_4b_fp8_scaled.safetensors',
        'sha256': '54bd5144df0bbc25dd6ccadfcb826b521445a1b06ae5a42570bdd2974ca87094',
    },
    {
        'path': 'vae/qwen_image_vae.safetensors',
        'size': 253806246,
        'url': 'https://huggingface.co/Comfy-Org/Krea-2/resolve/1171f4dc7e325bf3734586942e737872506c73e1/vae/qwen_image_vae.safetensors',
        'sha256': 'a70580f0213e67967ee9c95f05bb400e8fb08307e017a924bf3441223e023d1f',
    },
]

import requests
import shutil

def remaining_model_bytes(item):
    destination = APP / 'models' / item['path']
    if destination.is_file():
        return 0
    partial = destination.with_suffix(destination.suffix + '.partial')
    offset = partial.stat().st_size if partial.is_file() else 0
    return max(0, item['size'] - offset)

required_bytes = sum(remaining_model_bytes(item) for item in MODEL_MANIFEST) + 3 * 1024 ** 3
free_bytes = shutil.disk_usage(BASE).free
if free_bytes < required_bytes:
    raise RuntimeError(f'Need at least {required_bytes / 1024**3:.1f} GiB free for Krea2 files and temporary downloads; only {free_bytes / 1024**3:.1f} GiB available.')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download_verified(item):
    destination = APP / 'models' / item['path']
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file():
        actual = sha256_file(destination)
        if destination.stat().st_size == item['size'] and actual == item['sha256']:
            print('Verified existing file:', item['path'])
            return destination
        raise RuntimeError(f'Existing model file failed size/SHA-256 validation: {destination}')

    partial = destination.with_suffix(destination.suffix + '.partial')
    if partial.is_file() and partial.stat().st_size == item['size'] and sha256_file(partial) == item['sha256']:
        partial.replace(destination)
        print('Recovered verified partial file:', item['path'])
        return destination
    offset = partial.stat().st_size if partial.exists() else 0
    if offset >= item['size']:
        partial.unlink()
        offset = 0
    headers = {'Range': f'bytes={offset}-'} if offset else {}
    response = requests.get(item['url'], headers=headers, stream=True, timeout=(30, 120))
    response.raise_for_status()
    append = bool(offset and response.status_code == 206)
    if offset and not append:
        offset = 0
    mode = 'ab' if append else 'wb'
    digest = hashlib.sha256()
    if append:
        with partial.open('rb') as existing:
            for chunk in iter(lambda: existing.read(8 * 1024 * 1024), b''):
                digest.update(chunk)
    written = offset
    last_reported_gib = written // (1024 ** 3)
    with partial.open(mode) as handle:
        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                handle.write(chunk)
                digest.update(chunk)
                written += len(chunk)
                current_gib = written // (1024 ** 3)
                if current_gib > last_reported_gib:
                    print(f'Downloaded {current_gib} / {item["size"] // (1024 ** 3)} GiB: {item["path"]}', flush=True)
                    last_reported_gib = current_gib
    response.close()
    actual_hash = digest.hexdigest()
    if written != item['size'] or actual_hash != item['sha256']:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f'Model download failed validation for {item["path"]}: bytes={written}, sha256={actual_hash}')
    partial.replace(destination)
    print(f'Verified download: {item["path"]} ({written:,} bytes)')
    return destination

if MODE == 'tpu_experimental' and 'nvfp4' in MODEL_MANIFEST[0]['path'].lower():
    print('WARNING: the shared XLA patch does not enable NVFP4; continuing only as an explicitly unverified Krea2 compatibility probe. No precision fallback will be substituted.')

for model_item in MODEL_MANIFEST:
    download_verified(model_item)


## 6. Start ComfyUI on loopback

No public tunnel or external bind address is created. The server is used only by this notebook's local API requests.


In [ ]:
import pathlib
import socket

# Avoid mistaking an unrelated existing ComfyUI process for this TPU server.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe_socket:
    probe_socket.settimeout(2)
    if probe_socket.connect_ex(('127.0.0.1', 8188)) == 0:
        raise RuntimeError('Port 8188 is already in use. Stop the existing service or choose a free port before starting this notebook.')

LOG_PATH = APP / 'comfyui.log'
log_handle = LOG_PATH.open('a', encoding='utf-8')
cmd = [str(RUNTIME_PYTHON), 'main.py', '--listen', '127.0.0.1', '--port', '8188']
if MODE == 'tpu_experimental':
    cmd.extend(['--xla', '--cpu-vae'])
server_env = os.environ.copy()
if MODE == 'tpu_experimental':
    server_env['PJRT_DEVICE'] = 'TPU'
server = subprocess.Popen(cmd, cwd=str(APP), stdout=log_handle, stderr=subprocess.STDOUT, env=server_env)
BASE_URL = 'http://127.0.0.1:8188'

for _ in range(120):
    if server.poll() is not None:
        break
    try:
        response = requests.get(BASE_URL + '/system_stats', timeout=2)
        if response.ok:
            print('ComfyUI loopback API is ready; PID:', server.pid)
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    server.terminate()
    raise TimeoutError('ComfyUI did not become ready on 127.0.0.1:8188')

if server.poll() is not None:
    print(LOG_PATH.read_text(errors='replace')[-8000:])
    raise RuntimeError(f'ComfyUI exited with status {server.returncode}')


## 7. Queue one Krea 2 image through the local API

This uses the existing model filenames, Qwen Krea2 CLIP type, and Euler/simple sampling at 10 steps and CFG 1. No prompt-expansion service or LoRA is used.


In [ ]:
object_info = requests.get(BASE_URL + '/object_info', timeout=30).json()
required_nodes = ['UNETLoader', 'CLIPLoader', 'VAELoader', 'CLIPTextEncode', 'ConditioningZeroOut', 'EmptyLatentImage', 'KSampler', 'VAEDecode', 'SaveImage']
missing_nodes = [name for name in required_nodes if name not in object_info]
if missing_nodes:
    raise RuntimeError(f'Pinned ComfyUI is missing Krea2 workflow nodes: {missing_nodes}')

def combo_choices(node_name, input_name):
    value = object_info[node_name]['input']['required'][input_name]
    return value[0] if isinstance(value, list) and value and isinstance(value[0], list) else []

checks = [
    ('UNETLoader', 'unet_name', 'Neutrino_v2_base_nvfp4_svd.safetensors'),
    ('CLIPLoader', 'clip_name', 'qwen3vl_4b_fp8_scaled.safetensors'),
    ('VAELoader', 'vae_name', 'qwen_image_vae.safetensors'),
]
for node_name, input_name, filename in checks:
    if filename not in combo_choices(node_name, input_name):
        raise RuntimeError(f'ComfyUI did not register {filename} for {node_name}.{input_name}')
if 'krea2' not in combo_choices('CLIPLoader', 'type'):
    raise RuntimeError('This ComfyUI revision does not expose CLIPLoader type=krea2.')
print('ComfyUI registered the pinned Krea2 nodes, model files, and CLIP type.')

workflow = {
    '1': {'class_type': 'UNETLoader', 'inputs': {'unet_name': 'Neutrino_v2_base_nvfp4_svd.safetensors', 'weight_dtype': 'default'}},
    '2': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'qwen3vl_4b_fp8_scaled.safetensors', 'type': 'krea2', 'device': 'default'}},
    '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'qwen_image_vae.safetensors'}},
    '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': PROMPT, 'clip': ['2', 0]}},
    '5': {'class_type': 'ConditioningZeroOut', 'inputs': {'conditioning': ['4', 0]}},
    '6': {'class_type': 'EmptyLatentImage', 'inputs': {'width': WIDTH, 'height': HEIGHT, 'batch_size': 1}},
    '7': {'class_type': 'KSampler', 'inputs': {
        'seed': SEED, 'steps': STEPS, 'cfg': CFG, 'sampler_name': 'euler', 'scheduler': 'simple', 'denoise': 1.0,
        'model': ['1', 0], 'positive': ['4', 0], 'negative': ['5', 0], 'latent_image': ['6', 0],
    }},
    '8': {'class_type': 'VAEDecode', 'inputs': {'samples': ['7', 0], 'vae': ['3', 0]}},
    '9': {'class_type': 'SaveImage', 'inputs': {'images': ['8', 0], 'filename_prefix': 'Kaggle_Krea2_Neutrino'}},
}

client_id = str(uuid.uuid4())
queued = requests.post(BASE_URL + '/prompt', json={'prompt': workflow, 'client_id': client_id}, timeout=30)
if not queued.ok:
    raise RuntimeError(f'ComfyUI rejected the Krea2 workflow: {queued.status_code} {queued.text[:4000]}')
prompt_id = queued.json()['prompt_id']
print('Queued local Krea2 job:', prompt_id)

history = None
deadline = time.monotonic() + 60 * 60
while time.monotonic() < deadline:
    if server.poll() is not None:
        raise RuntimeError(f'ComfyUI exited during generation ({server.returncode}). Log tail:\n{LOG_PATH.read_text(errors="replace")[-8000:]}')
    result = requests.get(BASE_URL + '/history/' + prompt_id, timeout=15)
    result.raise_for_status()
    history_data = result.json()
    if prompt_id in history_data:
        history = history_data[prompt_id]
        status = history.get('status', {}).get('status_str')
        if status == 'error':
            raise RuntimeError('ComfyUI Krea2 job failed: ' + json.dumps(history.get('status', {}), ensure_ascii=False)[:8000])
        if status == 'success':
            break
    time.sleep(3)
else:
    raise TimeoutError(f'Krea2 generation timed out after 60 minutes. Inspect {LOG_PATH}')

image_entry = history.get('outputs', {}).get('9', {}).get('images', [None])[0]
if not image_entry:
    raise RuntimeError('Krea2 job completed without a SaveImage output.')
output_path = APP / 'output' / image_entry.get('subfolder', '') / image_entry['filename']
if not output_path.is_file():
    raise FileNotFoundError(f'ComfyUI reported an output that is missing: {output_path}')
from IPython.display import display
from PIL import Image
with Image.open(output_path) as image:
    image.verify()
with Image.open(output_path) as image:
    if image.size != (WIDTH, HEIGHT):
        raise RuntimeError(f'Generated image dimensions differ from request: {image.size}')
    display(image.copy())
print('Krea2 image decoded and validated:', output_path)


## 8. Review the ComfyUI log and stop the local process when done

A successful TPU smoke test or ComfyUI startup is not Krea2 generation proof. Check the generation cell output and the ComfyUI log for model-loading, XLA, sampler, and VAE errors. Stop the server to release the Kaggle runtime.


In [ ]:
if LOG_PATH.is_file():
    log_text = LOG_PATH.read_text(errors='replace')
    print(log_text[-12000:])
if server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=20)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait(timeout=10)
log_handle.close()
print('ComfyUI stopped:', server.poll())
